# NC-Musical: AI Music Transcription (Batch Colab GPU Pipeline)
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine directly inside Google Colab.

**Hardware Acceleration:** NVIDIA GPUs (T4, L4, A100)

**Key Features:**
- **Batch Transcription:** Process single or multiple YouTube URLs / uploaded files sequentially.
- **Original Song Naming:** Automatically names MIDI and chord files after the original YouTube song title or uploaded filename.
- **Dynamic Prompts:** No hardcoded links — easily paste links or be prompted on run.
- **Future-Proof yt-dlp:** Direct live master-branch builds for dependable extraction.
- **SoundFont Previews:** In-notebook audio playback via `MS Basic.sf3`.
- **Automated Chord Analysis:** Extracts and timestamps harmonic progressions using `music21`.
- **Automatic Downloads & ZIP:** Triggers immediate browser downloads and bundles batch results into `.zip`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [1]:
# @title 1. (Optional) Connect Google Drive for Persistent Model Storage
# @markdown Connecting Google Drive saves downloaded AI models and SoundFonts permanently so you don't re-download them across Colab restarts.
# @markdown Set **use_google_drive** to `False` to skip the authorization prompt and use fast session temporary storage instead.
use_google_drive = True # @param {type:"boolean"}
import os

if use_google_drive:
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive is already connected and active!")
    else:
        try:
            from google.colab import drive
            print("Connecting to Google Drive...")
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"Notice: Google Drive mount skipped ({e}). Falling back to local session storage.")

if os.path.exists('/content/drive/MyDrive'):
    drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
    os.makedirs(drive_models_dir, exist_ok=True)
    hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")
    print(f"✓ Persistent Model Storage enabled: {drive_models_dir}")
else:
    local_cache_dir = "/content/cache/huggingface"
    os.makedirs(local_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = local_cache_dir
    print("ℹ Session Storage active: Models will be cached in /content/cache/ for this session.")


Connecting to Google Drive...
Mounted at /content/drive
✓ Persistent Model Storage enabled: /content/drive/MyDrive/NC-Musical-Models


In [2]:
# @title 2. Check Hardware (GPU / TPU v5e-1), Tokens & Setup Repository
import os
import sys
import subprocess
import getpass
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Checking hardware accelerator environment...")
has_gpu = torch.cuda.is_available()
has_tpu = False

if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ NVIDIA GPU Detected: {gpu_name}")
    !nvidia-smi
else:
    # Check for Google Colab TPU v5e / TPU v2/v3 environment
    if 'COLAB_TPU_ADDR' in os.environ or os.environ.get('PJRT_DEVICE') == 'TPU' or os.path.exists('/dev/accel0'):
        has_tpu = True
        os.environ['PJRT_DEVICE'] = 'TPU'
        print('✓ Google Cloud TPU (v5e-1 / PJRT) Detected!')
    else:
        try:
            import torch_xla.core.xla_model as xm
            has_tpu = True
            print(f'✓ Google Cloud TPU Detected: {xm.xla_device()}')
        except Exception:
            print('ℹ Standard CPU environment detected (No GPU/TPU found).')

# Configure HuggingFace Token (HF_TOKEN)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    print("\nNotice: MuScriptor models on HuggingFace require a HuggingFace User Token (HF_TOKEN).")
    print("If saved in Colab Secrets as 'HF_TOKEN', it is loaded automatically. Otherwise, enter below:")
    try:
        hf_token = getpass.getpass("HuggingFace Token (HF_TOKEN): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace Access Token configured successfully!")

print("\nInstalling system dependencies (FFmpeg and FluidSynth)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth

print("\nCloning / Updating NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None

        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")

        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    !git pull || true
    print("\nInstalling Python dependencies (Future-proof live yt-dlp master branch)...")
    !pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" music21 soundfile mido
    if has_tpu:
        try:
            import torch_xla
        except ImportError:
            print("Installing torch_xla TPU support...")
            !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html
    !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


Checking hardware accelerator environment...
✓ NVIDIA GPU Detected: Tesla T4
Wed Aug 26 05:23:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                    

In [3]:
# @title 3. Setup MS Basic.sf3 SoundFont
import os
import shutil
import urllib.request

local_sf3_path = "/content/NC-Musical/MS Basic.sf3"
drive_sf3_path = "/content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3"
has_drive = os.path.exists("/content/drive/MyDrive")

target_sf3 = drive_sf3_path if has_drive else local_sf3_path

if not os.path.exists(target_sf3):
    print("Downloading MS Basic.sf3 SoundFont (~50MB)...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            os.makedirs(os.path.dirname(target_sf3), exist_ok=True)
            urllib.request.urlretrieve(url, target_sf3)
            print(f"✓ MS Basic.sf3 downloaded successfully to {target_sf3}!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print(f"✓ MS Basic.sf3 SoundFont found at: {target_sf3}")

# Sync to local workspace path if Drive was used
if has_drive and os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    shutil.copy2(drive_sf3_path, local_sf3_path)


✓ MS Basic.sf3 SoundFont found at: /content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3


In [ ]:
# @title 4. AI Music Transcription (Batch & Single Processing)
# @markdown ### 📥 Audio Input & Model Configuration
input_source = "Upload Audio/Video File(s)" # @param ["YouTube URL(s)", "Upload Audio/Video File(s)"]
youtube_urls = "" # @param {type:"string"}
model_size = "large" # @param ["small", "medium", "large"]

# @markdown ### 🎛️ Target Instrument Focus (Preset Mode)
# @markdown Choose which instrument(s) to isolate and transcribe from the mix:
instrument_focus = "All Instruments (Full Mix)" # @param ["All Instruments (Full Mix)", "Piano Only (Acoustic & Electric Keys)", "Guitars (Acoustic & Electric)", "Rhythm Section (Bass & Drums)", "Vocals & Lead Melody", "Strings & Orchestral", "Custom Selection (Configure Below)"]

# @markdown ### ⚙️ Advanced Custom Instrument Filters (Used if 'Custom Selection' chosen above)
focus_piano = True # @param {type:"boolean"}
focus_acoustic_guitar = False # @param {type:"boolean"}
focus_electric_guitar = False # @param {type:"boolean"}
focus_bass = False # @param {type:"boolean"}
focus_drums = False # @param {type:"boolean"}
focus_vocals = False # @param {type:"boolean"}
focus_strings = False # @param {type:"boolean"}
focus_brass = False # @param {type:"boolean"}
focus_woodwinds = False # @param {type:"boolean"}
focus_synths = False # @param {type:"boolean"}
custom_instruments_text = "" # @param {type:"string"}

# @markdown ### ⏱️ Musical Timing, Tempo & Meter Settings
# @markdown Configure how bar lines, tempo, and time signatures are mapped into the MIDI file:
tempo_mode = "Manual Fixed BPM (Recommended for DAWs)" # @param ["Manual Fixed BPM (Recommended for DAWs)", "Auto-Detect Dynamic (Fluctuating Tempo Map)", "Auto-Detect Fixed (Single Average BPM)", "Bypass (Default 120 BPM)"]
bpm = 120.0 # @param {type:"number"}
beat_scale = 1.0 # @param [0.5, 1.0, 2.0]
first_downbeat_sec = 0.0 # @param {type:"number"}

time_signature = "4/4 (Common Time)" # @param ["4/4 (Common Time)", "3/4 (Waltz)", "2/4 (March)", "6/8 (Compound Duple)", "12/8 (Compound Quadruple)", "5/4 (Odd Meter)", "7/8 (Balkan/Prog)", "Custom (Specify Below)"]
custom_beats_per_bar = 4 # @param {type:"integer"}
custom_time_sig_denom = 4 # @param {type:"integer"}

# @markdown ### 🎼 Key Signature Configuration
# @markdown Choose key signature to embed into MIDI and guide harmonic analysis:
key_signature = "Auto-Detect Harmonic Key (music21 / Krumhansl-Schmuckler)" # @param ["Auto-Detect Harmonic Key (music21 / Krumhansl-Schmuckler)", "C Major / A Minor", "G Major / E Minor (1#)", "D Major / B Minor (2#)", "A Major / F# Minor (3#)", "E Major / C# Minor (4#)", "B Major / G# Minor (5#)", "F# Major / D# Minor (6#)", "F Major / D Minor (1b)", "Bb Major / G Minor (2b)", "Eb Major / C Minor (3b)", "Ab Major / F Minor (4b)", "Db Major / Bb Minor (5b)", "Gb Major / Eb Minor (6b)", "Do Not Embed Key Signature"]

# @markdown ### 📐 Rhythmic Quantization & Tuplets Control
# @markdown Snap notes onto a clean musical grid and eliminate unwanted tuplets/triplets:
enable_quantization = True # @param {type:"boolean"}
quantization_grid = "No Tuplets (Straight 1/16 Sixteenth Notes - Recommended)" # @param ["No Tuplets (Straight 1/16 Sixteenth Notes - Recommended)", "No Tuplets (Straight 1/8 Eighth Notes)", "No Tuplets (Straight 1/32 Thirty-Second Notes)", "Eighth Triplets (Tuplets / Swing)", "Sixteenth Triplets (Tuplets)", "Auto Best-Fit (Adaptive Model Selection)"]
quantize_strength = 1.0 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
fill_gaps_beats = 0.25 # @param {type:"number"}
trim_leading_bars = False # @param {type:"boolean"}

# @markdown ### 📤 Export Options
export_chord_names = False # @param {type:"boolean"}
download_zip_for_batch = False # @param {type:"boolean"}

import os
import sys
import re
import io
import time
import math
import gc
import zipfile
import shutil
import subprocess
import torch
from pathlib import Path
from IPython.display import Audio, display
import google.colab.files

# Set PyTorch memory allocator to avoid fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Ensure yt-dlp, music21, and mido are present
!pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" music21 mido
import yt_dlp
import mido

def sanitize_filename(name: str) -> str:
    """Clean filename of illegal filesystem characters."""
    clean = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    clean = re.sub(r'\s+', " ", clean)
    return clean[:120] if clean else "transcription_result"

# ── 1. Resolve Target Instruments ─────────────────────────────────────────────
INST_ALIASES = {
    "piano": "acoustic_piano", "guitar": "acoustic_guitar",
    "electric_guitar": "clean_electric_guitar", "bass": "electric_bass",
    "drums": "drums", "drum": "drums", "percussion": "drums",
    "strings": "string_ensemble", "brass": "brass_section",
    "sax": "soprano_and_alto_sax", "saxophone": "soprano_and_alto_sax",
    "flute": "flutes", "synth": "synth_lead", "pad": "synth_pad",
    "vocal": "voice", "vocals": "voice",
}

inst_list = None
if instrument_focus == "All Instruments (Full Mix)":
    inst_list = None
elif instrument_focus == "Piano Only (Acoustic & Electric Keys)":
    inst_list = ["acoustic_piano", "electric_piano"]
elif instrument_focus == "Guitars (Acoustic & Electric)":
    inst_list = ["acoustic_guitar", "clean_electric_guitar", "distorted_electric_guitar"]
elif instrument_focus == "Rhythm Section (Bass & Drums)":
    inst_list = ["electric_bass", "acoustic_bass", "drums"]
elif instrument_focus == "Vocals & Lead Melody":
    inst_list = ["voice", "synth_lead"]
elif instrument_focus == "Strings & Orchestral":
    inst_list = ["string_ensemble", "brass_section", "flutes"]
elif instrument_focus == "Custom Selection (Configure Below)":
    selected = []
    if focus_piano: selected.extend(["acoustic_piano", "electric_piano"])
    if focus_acoustic_guitar: selected.append("acoustic_guitar")
    if focus_electric_guitar: selected.extend(["clean_electric_guitar", "distorted_electric_guitar"])
    if focus_bass: selected.extend(["electric_bass", "acoustic_bass"])
    if focus_drums: selected.append("drums")
    if focus_vocals: selected.append("voice")
    if focus_strings: selected.append("string_ensemble")
    if focus_brass: selected.append("brass_section")
    if focus_woodwinds: selected.append("flutes")
    if focus_synths: selected.extend(["synth_lead", "synth_pad"])
    if custom_instruments_text.strip():
        for raw_name in custom_instruments_text.split(","):
            k = raw_name.strip().lower().replace(" ", "_")
            if k: selected.append(INST_ALIASES.get(k, k))
    inst_list = list(dict.fromkeys(selected)) if selected else None

if inst_list:
    print(f"🎯 Target Instrument Focus: {inst_list}")
else:
    print("🎯 Target Instrument Focus: ALL Instruments (Full Multitrack Transcription)")

# ── 2. Resolve Timing, Meter, Key & Quantization Parameters ────────────────────
def _resolve_meter(ts_str, custom_num, custom_denom):
    if "custom" in ts_str.lower():
        return max(1, int(custom_num)), max(1, int(custom_denom))
    clean = ts_str.split("(")[0].strip()
    if "/" in clean:
        parts = clean.split("/")
        try:
            return int(parts[0].strip()), int(parts[1].strip())
        except ValueError:
            pass
    return 4, 4

eff_ts_num, eff_ts_denom = _resolve_meter(time_signature, custom_beats_per_bar, custom_time_sig_denom)

# Grid division mapping
# grid_div specifies subdivisions per beat:
# 4 = 1/16 sixteenths (no tuplets)
# 2 = 1/8 eighths (no tuplets)
# 8 = 1/32 thirty-seconds (no tuplets)
# 3 = eighth triplets (tuplets)
# 6 = sixteenth triplets (tuplets)
grid_div = 4
if "1/8" in quantization_grid:
    grid_div = 2
elif "1/32" in quantization_grid:
    grid_div = 8
elif "Eighth Triplets" in quantization_grid:
    grid_div = 3
elif "Sixteenth Triplets" in quantization_grid:
    grid_div = 6
else:
    grid_div = 4 # Default 1/16 sixteenth notes (No Tuplets)

is_no_tuplets = (grid_div in (1, 2, 4, 8, 16))

print(f"⏱️ Meter: {eff_ts_num}/{eff_ts_denom} | Tempo Mode: {tempo_mode} ({bpm} BPM, Scale: {beat_scale}x)")
print(f"📐 Quantization: {'Enabled' if enable_quantization else 'Disabled'} | Grid: 1/{grid_div*eff_ts_denom//4} ({'No Tuplets' if is_no_tuplets else 'Tuplets Allowed'}) | Strength: {quantize_strength:.2f}")

# ── 3. Embedded High-Performance MIDI Engine ──────────────────────────────────
_MAJOR_PROFILE = [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]
_MINOR_PROFILE = [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]
_MIDO_MAJOR_KEYS = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'F#', 'G', 'Ab', 'A', 'Bb', 'B']
_MIDO_MINOR_KEYS = ['Cm', 'C#m', 'Dm', 'Ebm', 'Em', 'Fm', 'F#m', 'Gm', 'G#m', 'Am', 'Bbm', 'Bm']
_KEY_PRESET_MAP = {
    "c major / a minor": "C", "g major / e minor (1#)": "G", "d major / b minor (2#)": "D",
    "a major / f# minor (3#)": "A", "e major / c# minor (4#)": "E", "b major / g# minor (5#)": "B",
    "f# major / d# minor (6#)": "F#", "f major / d minor (1b)": "F", "bb major / g minor (2b)": "Bb",
    "eb major / c minor (3b)": "Eb", "ab major / f minor (4b)": "Ab", "db major / bb minor (5b)": "Db",
    "gb major / eb minor (6b)": "Gb"
}

def detect_key_signature_from_notes(mid):
    """Accurately detect key signature using pitch histogram correlation."""
    try:
        from music21 import converter
        temp_buf = io.BytesIO()
        mid.save(file=temp_buf)
        temp_buf.seek(0)
        s = converter.parse(temp_buf.getvalue(), format='midi')
        k = s.analyze('key')
        tonic = k.tonic.name
        return tonic + ('m' if k.mode == 'minor' else '')
    except Exception:
        pass

    histogram = [0.0] * 12
    ticks_per_beat = mid.ticks_per_beat or 480
    for track in mid.tracks:
        active = {}
        curr = 0
        for msg in track:
            curr += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                active[msg.note] = curr
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                if msg.note in active:
                    start = active.pop(msg.note)
                    histogram[msg.note % 12] += max(1, curr - start) / ticks_per_beat

    if sum(histogram) == 0:
        return 'C'

    def corr(x, y):
        n = 12
        mx, my = sum(x) / n, sum(y) / n
        num = sum((x[i] - mx) * (y[i] - my) for i in range(n))
        den = math.sqrt(sum((x[i] - mx) ** 2 for i in range(n)) * sum((y[i] - my) ** 2 for i in range(n)))
        return num / den if den > 0 else 0

    best_score, best_key = -float('inf'), 'C'
    for r in range(12):
        maj_r = _MAJOR_PROFILE[-r:] + _MAJOR_PROFILE[:-r]
        min_r = _MINOR_PROFILE[-r:] + _MINOR_PROFILE[:-r]
        c_maj = corr(histogram, maj_r)
        if c_maj > best_score:
            best_score, best_key = c_maj, _MIDO_MAJOR_KEYS[r]
        c_min = corr(histogram, min_r)
        if c_min > best_score:
            best_score, best_key = c_min, _MIDO_MINOR_KEYS[r]
    return best_key

def postprocess_midi_events(
    mid,
    bpm_val=120.0,
    time_sig_num=4,
    time_sig_denom=4,
    key_sig_choice="auto",
    quantize=True,
    grid_division=4,
    quant_strength=1.0,
    fill_gaps=0.25,
    trim_lead_bars=False
):
    """Apply comprehensive musical post-processing: Time Signature, Key Signature, Tempo, and Quantization."""
    ticks_per_beat = mid.ticks_per_beat or 480
    ticks_per_beat_unit = int(round(ticks_per_beat * (4.0 / time_sig_denom))) if time_sig_denom else ticks_per_beat

    # Resolve Key Signature
    final_key = None
    k_lower = str(key_sig_choice).lower()
    if "auto" in k_lower:
        final_key = detect_key_signature_from_notes(mid)
    elif k_lower in _KEY_PRESET_MAP:
        final_key = _KEY_PRESET_MAP[k_lower]
    elif "do not embed" not in k_lower and key_sig_choice != "None":
        final_key = str(key_sig_choice).strip()

    # Apply Quantization
    if quantize:
        grid_ticks = max(1, int(round(ticks_per_beat_unit / grid_division)))
        fill_gap_ticks = int(round(fill_gaps * ticks_per_beat_unit)) if fill_gaps > 0 else 0
        ticks_per_bar = max(grid_ticks, int(round(ticks_per_beat_unit * time_sig_num)))

        shift_ticks = 0
        if trim_lead_bars:
            earliest = float('inf')
            for track in mid.tracks:
                curr = 0
                for msg in track:
                    curr += msg.time
                    if msg.type == 'note_on' and msg.velocity > 0:
                        if curr < earliest: earliest = curr
            if earliest != float('inf') and earliest >= ticks_per_bar:
                shift_bars = int(earliest // ticks_per_bar)
                shift_ticks = shift_bars * ticks_per_bar

        for t_idx, track in enumerate(mid.tracks):
            curr_tick = 0
            abs_events = []
            for msg in track:
                curr_tick += msg.time
                abs_events.append((curr_tick, msg))

            active_notes = {}
            note_pairs = []
            other_events = []

            for atick, msg in abs_events:
                if msg.type == 'note_on' and msg.velocity > 0:
                    active_notes.setdefault((msg.channel, msg.note), []).append((atick, msg))
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    k = (msg.channel, msg.note)
                    if k in active_notes and active_notes[k]:
                        start_tick, on_msg = active_notes[k].pop(0)
                        note_pairs.append({'channel': msg.channel, 'pitch': msg.note, 'start': start_tick, 'end': atick, 'on_msg': on_msg, 'off_msg': msg})
                    else:
                        other_events.append((atick, msg))
                else:
                    other_events.append((atick, msg))

            for k, unclosed in active_notes.items():
                for start_tick, on_msg in unclosed:
                    note_pairs.append({'channel': k[0], 'pitch': k[1], 'start': start_tick, 'end': start_tick + grid_ticks, 'on_msg': on_msg, 'off_msg': mido.Message('note_off', channel=k[0], note=k[1], velocity=0, time=0)})

            note_pairs.sort(key=lambda n: (n['channel'], n['pitch'], n['start']))

            if fill_gap_ticks > 0:
                for i in range(len(note_pairs) - 1):
                    cur_n, next_n = note_pairs[i], note_pairs[i + 1]
                    if cur_n['channel'] == next_n['channel'] and cur_n['pitch'] == next_n['pitch']:
                        gap = next_n['start'] - cur_n['end']
                        if 0 < gap <= fill_gap_ticks:
                            cur_n['end'] = next_n['start']

            quantized_events = []
            for n in note_pairs:
                orig_start, orig_dur = n['start'], max(1, n['end'] - n['start'])
                snapped_start = int(round(orig_start / grid_ticks)) * grid_ticks
                new_start = max(0, int(round(orig_start + (snapped_start - orig_start) * quant_strength)) - shift_ticks)
                snapped_dur = max(grid_ticks, int(round(orig_dur / grid_ticks)) * grid_ticks)
                new_dur = max(grid_ticks // 2, int(round(orig_dur + (snapped_dur - orig_dur) * quant_strength)))
                quantized_events.append((new_start, n['on_msg']))
                quantized_events.append((new_start + new_dur, n['off_msg']))

            for atick, msg in other_events:
                shifted = max(0, atick - shift_ticks) if atick > 0 else 0
                quantized_events.append((shifted, msg))

            def _ev_priority(item):
                tick, msg = item
                if getattr(msg, 'type', None) == 'end_of_track': return (tick, 999)
                if msg.is_meta: return (tick, 0)
                if msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0): return (tick, 1)
                if msg.type == 'note_on': return (tick, 2)
                return (tick, 3)

            quantized_events.sort(key=_ev_priority)

            new_track = mido.MidiTrack()
            prev_tick = 0
            for atick, msg in quantized_events:
                delta = max(0, atick - prev_tick)
                msg.time = delta
                new_track.append(msg)
                prev_tick = atick
            mid.tracks[t_idx] = new_track

    # Inject MetaMessages into Track 0
    tempo_track = mid.tracks[0] if mid.tracks else mido.MidiTrack()
    if not mid.tracks: mid.tracks.append(tempo_track)

    clean_track = [m for m in tempo_track if not (m.is_meta and m.type in ('time_signature', 'key_signature', 'set_tempo'))]
    tempo_track.clear()
    tempo_track.extend(clean_track)

    meta_msgs = []
    if time_sig_num and time_sig_denom:
        meta_msgs.append(mido.MetaMessage('time_signature', numerator=int(time_sig_num), denominator=int(time_sig_denom), clocks_per_click=24, notated_32nd_notes_per_beat=8, time=0))
    if final_key:
        try: meta_msgs.append(mido.MetaMessage('key_signature', key=str(final_key), time=0))
        except Exception: pass
    if bpm_val and bpm_val > 0:
        meta_msgs.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(float(bpm_val)), time=0))

    for m in reversed(meta_msgs):
        tempo_track.insert(0, m)

    out_buf = io.BytesIO()
    mid.save(file=out_buf)
    return mid, out_buf.getvalue(), final_key or "Unspecified"

# ── 4. Gather Items into Processing Queue ─────────────────────────────────────
queue = []
if input_source == "YouTube URL(s)":
    raw_input = youtube_urls.strip()
    if not raw_input:
        print("━" * 70)
        print("No YouTube URL provided in the form field.")
        raw_input = input("Please paste one or multiple YouTube URLs (separated by spaces or commas): ").strip()
        print("━" * 70 + "\n")
    urls = [u.strip() for u in re.split(r'[\n,\s]+', raw_input) if u.strip() and ("youtube.com" in u or "youtu.be" in u)]
    if not urls:
        print("❌ No valid YouTube URLs found. Please check your input and run again.")
    else:
        for url in urls:
            queue.append(("youtube", url, None))
else:
    print("Please click 'Choose Files' below to upload audio/video file(s) (MP3, WAV, MP4, M4A, FLAC):")
    uploaded = google.colab.files.upload()
    if uploaded:
        for fname in uploaded.keys():
            stem = Path(fname).stem
            queue.append(("file", fname, stem))
    else:
        print("❌ No files were uploaded.")

if not queue:
    print("Queue is empty. Nothing to transcribe.")
else:
    print(f"✓ Found {len(queue)} track(s) in queue.")

    # ── 5. Persistent Hardware & Model Management ──────────────────────────────
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"

    global _CACHED_MODEL, _CACHED_MODEL_SIZE, _CACHED_DEVICE
    if '_CACHED_MODEL' not in globals():
        _CACHED_MODEL = None
        _CACHED_MODEL_SIZE = None
        _CACHED_DEVICE = None

    from muscriptor.transcription_model import TranscriptionModel
    from muscriptor.utils.audio import _read_wav_file
    from muscriptor.events import ProgressEvent
    from muscriptor.utils.auralization import synthesize

    if _CACHED_MODEL is not None and _CACHED_MODEL_SIZE == model_size and _CACHED_DEVICE == device:
        print(f"✓ Reusing loaded '{model_size}' model on {dev_name} (0s reload time, 0MB extra VRAM).")
        model = _CACHED_MODEL
    else:
        if _CACHED_MODEL is not None:
            print(f"Releasing previous '{_CACHED_MODEL_SIZE}' model from VRAM...")
            del _CACHED_MODEL
            _CACHED_MODEL = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"\nLoading MuScriptor AI Model ('{model_size}' on {dev_name})...")
        model = TranscriptionModel.load_model(weights_path=model_size, device=device)
        _CACHED_MODEL = model
        _CACHED_MODEL_SIZE = model_size
        _CACHED_DEVICE = device
        print("✓ Model loaded successfully!\n")

    # YouTube download helper
    def fetch_youtube_audio(url, temp_stem="/content/yt_temp"):
        cookie_paths = ["/content/cookies.txt", "/content/drive/MyDrive/cookies.txt", "/content/drive/MyDrive/NC-Musical-Models/cookies.txt"]
        found_cookie = next((cp for cp in cookie_paths if os.path.exists(cp)), None)
        base_opts = {'format': 'bestaudio/best', 'outtmpl': f'{temp_stem}.%(ext)s', 'quiet': True, 'no_warnings': True, 'noplaylist': True}
        strategies = [
            ("Standard yt-dlp", {}),
            ("Android + Web client", {'extractor_args': {'youtube': {'player_client': ['android', 'web']}}}),
            ("iOS + Mobile Web client", {'extractor_args': {'youtube': {'player_client': ['ios', 'mweb']}}}),
            ("TV + Embedded client", {'extractor_args': {'youtube': {'player_client': ['tv_embedded', 'web']}}}),
        ]
        if found_cookie:
            strategies.append(("Standard yt-dlp + Cookies", {'cookiefile': found_cookie}))
            strategies.append(("Android client + Cookies", {'cookiefile': found_cookie, 'extractor_args': {'youtube': {'player_client': ['android', 'web']}}}))

        last_err = None
        title = "Unknown Song"
        for name, extra_opts in strategies:
            opts = {**base_opts, **extra_opts}
            try:
                for f in Path("/content").glob("yt_temp.*"):
                    try: os.remove(f)
                    except: pass
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(url, download=True)
                    if info and 'title' in info: title = info['title']
                dl_files = list(Path("/content").glob("yt_temp.*"))
                if dl_files: return str(dl_files[0]), title, None
            except Exception as e:
                last_err = e
        return None, title, last_err

    def _fmt_time(s):
        return f"{int(s//60)}m {int(s%60):02d}s" if s >= 60 else f"{s:.0f}s"

    def _draw_progress(completed, total, elapsed, note_count):
        pct = int(completed / total * 100) if total > 0 else 0
        bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
        eta = (elapsed / max(completed, 1)) * (total - completed) if completed > 0 else 0
        print(f"\r  [{bar}] {pct:3d}%  |  Elapsed: {_fmt_time(elapsed)}  |  ETA: {_fmt_time(eta)}  |  Notes: {note_count}    ", end="", flush=True)

    # ── 6. Execute Batch Queue with Fail-Safe AI & Musical Post-Processing ─────
    batch_results = []
    output_dir = Path("/content/NC_Musical_Outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    for idx, (itype, isrc, ititle) in enumerate(queue, 1):
        print("═" * 70)
        print(f"🎵 [Track {idx}/{len(queue)}] Processing: {isrc if itype == 'youtube' else isrc}")
        print("═" * 70)

        wav_input = "/content/current_input.wav"
        if os.path.exists(wav_input):
            os.remove(wav_input)

        song_title = ititle
        if itype == "youtube":
            print("Downloading YouTube audio...")
            raw_audio, fetched_title, err = fetch_youtube_audio(isrc)
            if not raw_audio:
                print(f"❌ Failed to download audio for {isrc}: {err}")
                continue
            song_title = sanitize_filename(fetched_title)
            print(f"✓ Title: {song_title}")
            print("Converting to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(raw_audio), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])
            try: os.remove(raw_audio)
            except: pass
        else:
            song_title = sanitize_filename(ititle)
            print(f"✓ Title: {song_title}")
            print(f"Converting {isrc} to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(isrc), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])

        if not os.path.exists(wav_input):
            print(f"❌ Failed to prepare audio for {song_title}.")
            continue

        with open(wav_input, "rb") as f:
            wav, sr = _read_wav_file(f)

        print("Transcribing audio notes...")
        events = []
        start_t = time.time()
        devnull = open(os.devnull, 'w')
        real_stdout = sys.stdout
        try:
            with torch.inference_mode():
                for ev in model.transcribe((wav, sr), instruments=inst_list, batch_size=1):
                    sys.stdout = real_stdout
                    if isinstance(ev, ProgressEvent):
                        if ev.total > 0:
                            _draw_progress(ev.completed, ev.total, time.time() - start_t, len(events))
                    else:
                        events.append(ev)
                    sys.stdout = devnull
        finally:
            sys.stdout = real_stdout
            devnull.close()

        elapsed = time.time() - start_t
        print(f"\n\n✅ Complete in {_fmt_time(elapsed)} — {len(events)} note events extracted.")

        # Resolve tempo & beat grid with graceful fail-safe handling
        final_bpm = float(bpm) * float(beat_scale)
        if "Auto-Detect" in tempo_mode:
            try:
                # Attempt AI tempo / beat-tracking if available
                print("Attempting AI beat tracking & tempo detection...")
                if hasattr(model, 'transcribe_and_postprocess'):
                    _, ai_grid = model.transcribe_and_postprocess(wav_input, detect_tempo="best-effort", quantize=False)
                    if ai_grid is not None and getattr(ai_grid, 'bpm', None):
                        final_bpm = float(ai_grid.bpm) * float(beat_scale)
                        print(f"✓ AI Detected Tempo: {final_bpm:.1f} BPM")
            except Exception as tempo_err:
                print(f"⚠️ AI beat-tracking notice (long audio or memory limit): {tempo_err}")
                print(f"✓ Applied safe fallback tempo: {final_bpm:.1f} BPM ({eff_ts_num}/{eff_ts_denom})")

        # Convert raw notes to MIDI bytes
        raw_midi_bytes = model.events_to_midi_bytes(iter(events))
        raw_mid = mido.MidiFile(file=io.BytesIO(raw_midi_bytes))

        # Apply robust musical post-processing (Time Signature, Key Signature, Tempo & Quantization)
        print("Applying post-processing (Time Signature, Key Signature, Tempo & Quantization)...")
        processed_mid, final_midi_bytes, resolved_key = postprocess_midi_events(
            raw_mid,
            bpm_val=final_bpm,
            time_sig_num=eff_ts_num,
            time_sig_denom=eff_ts_denom,
            key_sig_choice=key_signature,
            quantize=enable_quantization,
            grid_division=grid_div,
            quant_strength=quantize_strength,
            fill_gaps=fill_gaps_beats,
            trim_lead_bars=trim_leading_bars
        )

        midi_path = output_dir / f"{song_title}.mid"
        with open(midi_path, "wb") as f:
            f.write(final_midi_bytes)

        print(f"💾 MIDI Saved: {midi_path.name}")
        print(f" • Time Signature: {eff_ts_num}/{eff_ts_denom} | Key: {resolved_key} | Tempo: {final_bpm:.1f} BPM")
        print(f" • Quantization:   {'Enabled (1/' + str(grid_div*eff_ts_denom//4) + ' - ' + ('No Tuplets' if is_no_tuplets else 'Tuplets') + ')' if enable_quantization else 'Disabled'}")

        # Chord Analysis with original song title & musical metadata header
        chord_path = None
        if export_chord_names:
            chord_path = output_dir / f"{song_title}_chords.txt"
            try:
                from collections import defaultdict
                from music21 import converter, chord as m21chord
                score = converter.parse(str(midi_path))

                time_groups = defaultdict(list)
                for el in score.flatten().notes:
                    start = float(el.offset)
                    end = start + float(el.quarterLength)
                    pitches = [el.pitch] if el.isNote else el.pitches
                    for p in pitches:
                        time_groups[start].append((1, p))
                        time_groups[end].append((-1, p))

                lines = [
                    f"CHORD ANALYSIS — {song_title}",
                    f"Key Signature: {resolved_key} | Time Signature: {eff_ts_num}/{eff_ts_denom} | Tempo: {final_bpm:.1f} BPM",
                    "=" * 60
                ]
                active_pitches = defaultdict(int)
                prev_label = None
                prev_pitch_set = None

                for t in sorted(time_groups.keys()):
                    for event_type, p in sorted(time_groups[t], key=lambda x: x[0]):
                        if event_type == -1:
                            active_pitches[p] -= 1
                            if active_pitches[p] <= 0:
                                del active_pitches[p]
                        elif event_type == 1:
                            active_pitches[p] += 1

                    if active_pitches:
                        current_pitch_set = frozenset(active_pitches.keys())
                        if current_pitch_set != prev_pitch_set:
                            c = m21chord.Chord(sorted(list(current_pitch_set)))
                            ts = f"{int(t//60)}:{t%60:05.2f}"
                            try:
                                label = f"{c.root().name} {c.commonName}" if c.root() else " ".join(p.name for p in c.pitches)
                            except Exception:
                                label = " ".join(p.name for p in c.pitches)
                            if label != prev_label:
                                lines.append(f"[{ts}]  {label}")
                                prev_label = label
                            prev_pitch_set = current_pitch_set

                with open(chord_path, "w", encoding="utf-8") as f:
                    f.write("\n".join(lines))
                print(f"🎼 Chords Saved: {chord_path.name}")
            except Exception as e:
                print(f"Chord analysis notice: {e}")

        # Audio preview
        preview_wav = "/content/preview.wav"
        sf3_path = "/content/NC-Musical/MS Basic.sf3"
        try:
            synthesize(str(midi_path), preview_wav, soundfont_path=sf3_path if os.path.exists(sf3_path) else None)
            print(f"🎧 Audio Preview ({song_title}):")
            display(Audio(preview_wav))
        except Exception:
            pass

        # Auto download individual files
        try:
            google.colab.files.download(str(midi_path))
            if chord_path and chord_path.exists():
                google.colab.files.download(str(chord_path))
        except Exception as e:
            print(f"Download notice: {e}")

        batch_results.append({
            "title": song_title,
            "midi": str(midi_path),
            "chords": str(chord_path) if chord_path and chord_path.exists() else None,
            "notes": len(events),
            "key": resolved_key,
            "meter": f"{eff_ts_num}/{eff_ts_denom}",
            "bpm": f"{final_bpm:.1f}",
            "time": _fmt_time(elapsed)
        })

        del wav, events
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── 7. Batch ZIP Package ──────────────────────────────────────────────────
    if len(batch_results) > 1 and download_zip_for_batch:
        zip_path = "/content/NC_Musical_Batch_Results.zip"
        print("\n" + "═" * 70)
        print("📦 Packaging all batch transcriptions into ZIP archive...")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for item in batch_results:
                if os.path.exists(item['midi']):
                    zipf.write(item['midi'], arcname=Path(item['midi']).name)
                if item['chords'] and os.path.exists(item['chords']):
                    zipf.write(item['chords'], arcname=Path(item['chords']).name)
        print(f"✓ ZIP archive created: {zip_path}")
        google.colab.files.download(zip_path)

    # ── 8. Summary Table ──────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("📊 BATCH TRANSCRIPTION SUMMARY")
    print("═" * 70)
    for r in batch_results:
        print(f" • {r['title']} | Notes: {r['notes']:,} | Key: {r['key']} | Meter: {r['meter']} | Tempo: {r['bpm']} BPM | Time: {r['time']}")
    print(f"\nAll results saved to directory: {output_dir}")
